In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Efficient Convnets

The networks of the previous sections were designed to win accuracy
benchmarks, and they were trained and run on datacenter GPUs. Many
convolutional networks run elsewhere: on phones, cameras, cars, and embedded
boards. There latency, memory, and energy are fixed budgets, and a network
that exceeds them cannot ship, however accurate it is. A phone that
classifies every frame its camera records is a simple example. VGG-16 needs
15.3 billion multiply-adds to classify one $224 \times 224$ image, so at 30
frames a second the phone would have to perform 460 billion multiply-adds per
second. The network's 138 million parameters would occupy 550 MB in single
precision, in memory next to every other app. How can we make a
convolutional network an order of magnitude cheaper, and what does that cost
in accuracy?

This section studies three ideas that make convnets cheaper.
*Factorization* replaces each dense convolution by a per-channel spatial
filter and a $1 \times 1$ channel mixer. *Scaling and search* fixes the block
and lets an optimizer choose the widths, depths, and input resolution.
*Re-parameterization* trains a network with several branches per layer and
then merges the branches into a single convolution that computes the same
function. We also measure what the first idea delivers, in accuracy and in
time. The answer is less flattering than the count of multiply-adds, and it
depends on the task.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
import time

## Depthwise-Separable Networks

Most of the arithmetic of a convnet is spent in its dense convolutions, so
the first idea is to make each of them cheaper.

### Depthwise-Separable Convolutions

The cost of a dense convolution grows with the product $c_\textrm{i} c_\textrm{o}$
because every output channel reads every input channel. The grouped
convolution of ResNeXt (that section) relaxes this: it splits the
channels into $g$ groups and lets each output group read only its own input
group, which divides parameters and multiply-adds by $g$.
the figure shows which input channels each output channel
reads: the dense pattern becomes block diagonal.

![Input channels read by each output channel, for 8 input and 8 output channels. Each shaded cell is one $k_\textrm{h}\times k_\textrm{w}$ kernel; $g$ groups keep $1/g$ of them.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-groups.svg)

Let us take grouping to its extreme. With
$g = c_\textrm{i} = c_\textrm{o}$, every group holds one channel. This
*depthwise convolution* filters each channel with its own $k\times k$ kernel
and mixes no channels at all, the opposite of the $1\times 1$ convolution
(that section), which mixes channels and ignores neighbors. Each
does half of what a dense convolution does, and together they do both: a
depthwise $k\times k$ convolution followed by a *pointwise* $1\times 1$
convolution is a *depthwise-separable convolution*
[@chollet2017xception; @howard2017mobilenet].
the figure contrasts it with the dense layer it replaces:
the depthwise stage looks at neighborhoods within each channel, and the
pointwise stage recombines the channels at each position.

![Three input and three output channels with $3\times 3$ kernels. Top: a dense convolution, in which every output channel reads every input channel through a kernel stack of its own. Bottom: a depthwise convolution filters each channel separately, once for all output channels, and a pointwise convolution mixes the channels with one $1\times 1$ kernel stack per output channel.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-conv-depthwise.svg)

For a $k\times k$ kernel and an $h\times w$ output, the dense convolution
performs $hw\cdot c_\textrm{i} c_\textrm{o} k^2$ multiply-adds. The separable
pair performs $hw\cdot c_\textrm{i} k^2$ in the depthwise stage and
$hw\cdot c_\textrm{i} c_\textrm{o}$ in the pointwise stage, so the cost
relative to the dense layer is

$$
\frac{h w \cdot c_\textrm{i} k^2 + h w \cdot c_\textrm{i} c_\textrm{o}}{h w \cdot c_\textrm{i} c_\textrm{o} k^2}
= \frac{1}{c_\textrm{o}} + \frac{1}{k^2}.
$$

The same ratio holds for the weights. In the figure, with
$c_\textrm{i} = c_\textrm{o} = k = 3$, the dense layer has $3 \cdot 27 = 81$
weights and the separable pair $27 + 9 = 36$, a ratio of $1/3 + 1/9 = 4/9$.
Each further output channel costs the dense layer another 27 weights and the
separable pair another 3. For $k = 3$ and many output channels the ratio is
close to $1/9$: the separable layer is nearly nine times cheaper.

The savings come from a restriction. The separable pair applies the
depthwise filter $\mathbf{D}_c$ to input channel $c$ and scales the result by
the pointwise weight $P_{o,c}$ before adding it to output channel $o$. It is
therefore the dense convolution with kernel
$\mathbf{K}_{o,c} = P_{o,c}\,\mathbf{D}_c$: all output channels see the same
spatial filter of each input channel, only scaled, while a dense layer may
use a different filter for every pair $(o, c)$. Efficient architectures accept
this restriction and use the savings for more channels or more layers, as
the rest of this section shows.

We check the arithmetic with a dense $3\times 3$ convolution from 128 to 128
channels and its depthwise-separable factorization, without biases. Both map
the input to an output of the same shape, and
the equation predicts a ratio of parameter counts of
$(1/128 + 1/9)^{-1} \approx 8.4$.

Keras layers take channels-last inputs, so `X` below has shape
$(1, 32, 32, c_\textrm{i})$.

In [ ]:
c_i, c_o, k = 128, 128, 3
X = d2l.normal((1, 32, 32, c_i), 0, 1)
dense = tf.keras.layers.Conv2D(c_o, kernel_size=k, padding='same',
                               use_bias=False)
depthwise = tf.keras.layers.DepthwiseConv2D(kernel_size=k, padding='same',
                                            use_bias=False)
pointwise = tf.keras.layers.Conv2D(c_o, kernel_size=1, use_bias=False)
assert dense(X).shape == pointwise(depthwise(X)).shape
p_dense = dense.count_params()
p_sep = depthwise.count_params() + pointwise.count_params()
p_dense, p_sep, p_dense / p_sep

The dense layer has 147,456 weights and the separable pair 17,536, a ratio of
8.4 as predicted.

Keras has a separate `DepthwiseConv2D` layer; grouped convolutions use the
`groups` argument of `Conv2D`.

### MobileNet

MobileNet [@howard2017mobilenet] turned this layer into a whole
network. Every layer after the first is the block of
the figure: a depthwise $3 \times 3$ convolution and a
pointwise $1 \times 1$ convolution, each followed by batch normalization and
a ReLU. The network downsamples by giving the depthwise convolution a stride
of 2, so it needs no pooling layers inside, and it ends with global average
pooling and one linear layer, as NiN does (that section). This answers
our opening question rather well. On ImageNet, MobileNet comes within one
point of VGG-16's accuracy (70.6% against 71.5%) with 0.57 billion
multiply-adds and 4.2 million parameters: 27 times less arithmetic and 33
times fewer parameters.

![A depthwise-separable block from $c_\textrm{i}$ to $c_\textrm{o}$ channels. The depthwise $3 \times 3$ convolution filters each input channel separately; the pointwise $1 \times 1$ convolution mixes the channels at each position; batch normalization and a ReLU follow each convolution.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-dws-block.svg)

### The Inverted Bottleneck

The bottleneck blocks of ResNet and ResNeXt (that section)
compress: a $1 \times 1$ convolution reduces a wide representation, the
$3 \times 3$ convolution works in the narrow space, and a final $1 \times 1$
convolution expands it back. This is sensible as long as the $3 \times 3$
convolution is dense and therefore expensive. A depthwise convolution is
cheap even on wide tensors, and it has a reason to want them: it learns one
spatial filter per channel, so the number of filters in a block equals the
width at which the depthwise layer runs.

MobileNetV2 [@sandler2018mobilenetv2] therefore *inverts* the
bottleneck. The residual stream stays thin, say 24 channels. A $1 \times 1$
convolution expands it sixfold to 144 channels, the depthwise $3 \times 3$
convolution filters in the wide space, and a $1 \times 1$ projection returns
to 24 channels. No ReLU follows the projection: applied to so thin a
representation, a ReLU would zero the negative coordinates and with them
information that later layers cannot recover. the figure
places the two blocks side by side.

![A ResNet bottleneck block (left) and a MobileNetV2 inverted bottleneck (right); the input is at the bottom. The bottleneck narrows 256 channels to 64 for a dense $3 \times 3$ convolution and widens them again. The inverted bottleneck widens 24 channels to 144 for a depthwise $3 \times 3$ convolution, which learns one filter per channel, and projects back to 24 without a ReLU; the residual connection joins the thin ends.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-inverted-bottleneck.svg)

The residual connection joins the thin ends, so the tensors that stay in
memory from one block to the next are the small ones. This matters on a
phone, where activation memory rather than parameter storage is often the
binding constraint. The same block shape, with a $7 \times 7$ depthwise
convolution moved to the front, reappears in ConvNeXt
(that section).

### A Mini-MobileNet

We build a small MobileNetV1-style network for Fashion-MNIST. We use the
original block rather than the inverted bottleneck because it shows the
factorization most directly. The function below builds the pair of
the figure; a stride of 2 downsamples. Every convolution in the
networks of this section is followed by batch normalization, which has its
own shift, so the convolutions carry no bias. All convolution and linear
weights start from He initialization (that section).

In [ ]:
def he_init():
    """Normal weights with variance 2/n_in; one instance per layer."""
    return tf.keras.initializers.VarianceScaling(2.0, 'fan_in',
                                                 'untruncated_normal')

def dws_block(out_channels, strides=1):
    """Depthwise 3x3 and pointwise 1x1 convolutions, each with BN and ReLU."""
    return tf.keras.Sequential([
        tf.keras.layers.ZeroPadding2D(1),  # Pad 1 on every side
        tf.keras.layers.DepthwiseConv2D(  # Each unit sees n_in = 3 x 3 inputs
            3, strides, use_bias=False,
            depthwise_initializer=tf.keras.initializers.RandomNormal(
                stddev=math.sqrt(2 / 9))),
        tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv2D(out_channels, 1, use_bias=False,
                               kernel_initializer=he_init()),
        tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
        tf.keras.layers.ReLU()])

Like VGG (that section), the whole network is a list of block parameters,
here (channels, stride) pairs. A $3 \times 3$ convolution with stride 2 lifts
the grayscale input to 32 channels at half resolution; seven
depthwise-separable blocks then double the channels three times while halving
the feature map three times; global average pooling and a linear layer produce
the class scores.

The optimizer is new to this chapter. We train with stochastic gradient
descent with momentum 0.9, which steps along an exponentially weighted sum
of recent gradients instead of the latest gradient alone, so that its steps
are up to ten times longer than plain gradient steps at the same rate
(that section). The accuracies below are therefore not comparable
with those of ResNet-18 in that section, which was trained with
plain SGD. The learning rate again decays to zero along a half cosine
(that section), here without warmup and with the scheduler that
each framework provides: it starts at `lr` and reaches zero after the
`num_steps` minibatches of training (4,690 for 10 epochs).

In [ ]:
class MiniMobileNet(d2l.Classifier):
    def __init__(self, arch=((64, 1), (128, 2), (128, 1), (256, 2), (256, 1),
                             (512, 2), (512, 1)),
                 lr=0.05, num_steps=4690, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = tf.keras.Sequential([
            tf.keras.layers.ZeroPadding2D(1),
            tf.keras.layers.Conv2D(32, 3, 2, use_bias=False,
                                   kernel_initializer=he_init()),
            tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
            tf.keras.layers.ReLU()])
        for c_out, strides in arch:
            self.net.add(dws_block(c_out, strides))
        self.net.add(tf.keras.layers.GlobalAvgPool2D())
        self.net.add(tf.keras.layers.Dense(num_classes,
                                           kernel_initializer=he_init()))

    def configure_optimizers(self):
        schedule = tf.keras.optimizers.schedules.CosineDecay(self.lr,
                                                             self.num_steps)
        return tf.keras.optimizers.SGD(schedule, momentum=0.9)

How many parameters does the network have, and how many would it need with
dense convolutions? Counting the trainable parameters answers the first
question; we save the function `num_params` for later sections.

In [ ]:
def num_params(model):
    """The number of trainable parameters."""
    return sum(int(tf.size(w)) for w in model.trainable_variables)

model = MiniMobileNet()
assert model(tf.zeros((1, 96, 96, 1))).shape == (1, 10)
num_params(model)

The network has 542,474 parameters. A twin that replaced each separable pair by
one dense $3 \times 3$ convolution between the same channel counts would need
$9 \sum_l c_{\textrm{i},l}\, c_{\textrm{o},l} = 4{,}663{,}296$ weights in its
seven blocks alone, 8.8 times the 530,528 weights of the separable
convolutions. With up to 512 output channels, the network comes close to the
limit of 9.

### Arithmetic, Accuracy, and Time

Does the factorization give up accuracy? To find out, we compare the
mini-MobileNet with a dense VGG-style network (that section) that has
about the *same number of parameters*. A dense $3 \times 3$ layer between the
same two widths has about nine times as many weights, so with the same budget
the dense network stops at 128 channels where the separable one reaches 512.
Both networks use batch normalization after every convolution, the same
initialization, the same global-average-pooling head, the same optimizer, and
the same training budget. The comparison therefore matches parameters, not
architectures: the separable block and the wider channels it allows change
together.

In [ ]:
class VGGSmall(d2l.Classifier):
    """A dense VGG-style network with the same parameter budget."""
    def __init__(self, arch=((2, 32), (2, 64), (2, 128), (2, 128)),
                 lr=0.05, num_steps=4690, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = tf.keras.Sequential()
        for num_convs, c_out in arch:
            for _ in range(num_convs):
                self.net.add(tf.keras.layers.Conv2D(
                    c_out, 3, padding='same', use_bias=False,
                    kernel_initializer=he_init()))
                self.net.add(tf.keras.layers.BatchNormalization(
                    momentum=0.9, epsilon=1e-5))
                self.net.add(tf.keras.layers.ReLU())
            self.net.add(tf.keras.layers.MaxPool2D(2))
        self.net.add(tf.keras.layers.GlobalAvgPool2D())
        self.net.add(tf.keras.layers.Dense(num_classes,
                                           kernel_initializer=he_init()))

    def configure_optimizers(self):
        schedule = tf.keras.optimizers.schedules.CosineDecay(self.lr,
                                                             self.num_steps)
        return tf.keras.optimizers.SGD(schedule, momentum=0.9)

Before training, we count the arithmetic. A convolution with "same"
padding and stride $s$ turns an $h \times h$ input into an output of side
$\lceil h/s \rceil$, and each output position of each output channel costs
$k^2 c_\textrm{i} / g$ multiply-adds for a convolution with $g$ groups. The
functions below add these costs up for both networks on $96 \times 96$
inputs, including the linear head.

In [ ]:
def conv_macs(h, c_in, c_out, k=3, stride=1, groups=1):
    """Output side and multiply-adds of a k x k convolution, 'same' padding."""
    h = -(-h // stride)  # ceil(h / stride)
    return h, h * h * k * k * c_in * c_out // groups

def mobilenet_macs(arch, h=96, num_classes=10):
    h, total = conv_macs(h, 1, 32, stride=2)  # The stem
    c = 32
    for c_out, stride in arch:
        h, depthwise = conv_macs(h, c, c, stride=stride, groups=c)
        h, pointwise = conv_macs(h, c, c_out, k=1)
        total, c = total + depthwise + pointwise, c_out
    return total + c * num_classes

def vgg_macs(arch, h=96, num_classes=10):
    total, c = 0, 1
    for num_convs, c_out in arch:
        for _ in range(num_convs):
            h, macs = conv_macs(h, c, c_out)
            total, c = total + macs, c_out
        h //= 2  # 2 x 2 max-pooling
    return total + c * num_classes

narrow = ((2, 11), (2, 23), (2, 46), (2, 46))  # VGGSmall's widths / 2.8
macs = {'Mini-MobileNet': mobilenet_macs(MiniMobileNet().arch),
        'VGG-style': vgg_macs(VGGSmall().arch),
        'VGG-style, narrow': vgg_macs(narrow)}
macs, macs['VGG-style'] / macs['Mini-MobileNet']

At about the same parameter count, the dense network performs 385 million
multiply-adds per image and the separable one 50 million, 7.7 times fewer.
The dense network performs most of them in its first three stages, on
feature maps of 96 to 24 pixels on a side, where each weight is applied at
thousands of positions. MobileNet holds nearly three quarters of its weights
in the pointwise convolutions of its last two blocks, which see only
$6 \times 6$ feature maps.

The comparison that matters on a phone is the one at equal arithmetic. With
MobileNet's 50 million multiply-adds, the dense network has to shrink every
width by a factor of about 2.8, to 11, 23, 46, and 46 channels (the tuple
`narrow` above). That leaves it 76,000 parameters, a seventh of MobileNet's.
We train all three networks for 10 epochs on Fashion-MNIST, upsampled to
$96 \times 96$ pixels as for the networks of the previous sections.

In [ ]:
data = d2l.FashionMNIST(batch_size=128, resize=(96, 96))
num_steps = 10 * len(data.train_dataloader())  # 10 epochs of 469 steps
models = {'Mini-MobileNet': MiniMobileNet(lr=0.05, num_steps=num_steps),
          'VGG-style': VGGSmall(lr=0.05, num_steps=num_steps),
          'VGG-style, narrow': VGGSmall(narrow, lr=0.05, num_steps=num_steps)}
accuracy = {}
for name, model in models.items():
    trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
    trainer.fit(model, data)
    accuracy[name] = trainer.history['val_acc'][-1]

Multiply-adds count arithmetic, not time. To see how they translate into
time, we measure how long one forward pass takes on a batch of 128 images,
the work of inference, on the GPU. As in that section,
we run the computation once to warm up, repeat it, and wait for the GPU to
finish before reading the clock.

In [ ]:
def forward_ms(model, num_reps=50):
    """Milliseconds per forward pass of a batch of 128 images."""
    X = tf.random.normal((128, 96, 96, 1))
    @tf.function
    def forward(X):
        return model(X, training=False)
    forward(X).numpy()  # Trace, warm up, and wait
    start = time.perf_counter()
    for _ in range(num_reps):
        Y = forward(X)
    Y.numpy()  # Wait for the GPU
    return 1000 * (time.perf_counter() - start) / num_reps

for name, model in models.items():
    print(f'{name}: {num_params(model):,} parameters, '
          f'{macs[name] / 1e6:.1f}M multiply-adds, '
          f'accuracy {accuracy[name]:.3f}, '
          f'{forward_ms(model):.1f} ms/batch')

the table collects the results. The accuracies are the range
over repeated runs, and the time ratio is the range of each dense network's
forward time divided by the mini-MobileNet's.

:Mini-MobileNet against a dense VGG-style network with about the same number
of parameters and a narrow one with about the same number of multiply-adds
(Fashion-MNIST at $96 \times 96$, 10 epochs; validation accuracy and
forward-pass time ratio over repeated runs).

| Model | Parameters | Multiply-adds | Validation accuracy | Forward time |
|---|---:|---:|---:|---:|
| Mini-MobileNet | 542,474 | 50.3 million | 91–92% | 1 |
| VGG-style | 583,594 | 384.9 million | 93.5–94% | 2–3.5 |
| VGG-style, narrow | 75,854 | 49.1 million | 93% | 0.5–0.8 |

At the same parameter budget the dense network is about two points more
accurate, and it performs 7.7 times as much arithmetic to get there. The
first half of this sentence should not surprise us. The separable layer is a
restricted dense layer (that section), and the dense
network applies each of its weights at far more positions: it keeps
$96 \times 96$ feature maps for its first two convolutions, where MobileNet
halves the resolution in its stem. Equal parameters are not equal compute.

At equal arithmetic we might expect the comparison to turn around, since
this is the budget MobileNet was designed for. It does not. The narrow dense
network, with a seventh of MobileNet's parameters, is still about a point
more accurate, and its forward pass is faster. On this
task, narrowing a dense network is a better way to save arithmetic than
factorizing it. Why? By the equation the factorization
saves the most where the layers are wide: at 11 output channels it saves a
factor of 5, at 512 nearly 9. Fashion-MNIST's ten classes of small grayscale
images are served well by narrow layers. MobileNet was designed for
ImageNet, where its layers are up to 1024 channels wide and a dense network
of equal arithmetic would have to be very narrow indeed.

Between MobileNet and the dense network of equal parameters, the time
differs by much less than the arithmetic: a factor of 2 to 3.5, where the
multiply-adds differ by 7.7. A dense $3 \times 3$ convolution with
$c_\textrm{o}$ output channels performs $9 c_\textrm{o}$ multiply-adds for
every input value it reads, enough to keep a GPU's arithmetic units busy. A
depthwise convolution performs 9, and a GPU then spends most of the layer's
time moving activations to and from memory. Operation counts are only a
proxy for time, and how good a proxy depends on the hardware.

## Scaling and Searching

MobileNet fixes a block and chooses the channel counts by hand. Two
questions remain: how should a network grow when more computation is
available, and who should choose the width, depth, and resolution of each
stage?

### Compound Scaling

For the first question it helps to count. The cost of a convnet grows
linearly in its depth, quadratically in its width (each layer costs
$c_\textrm{i} c_\textrm{o}$), and quadratically in the input resolution
(each layer costs $hw$). Scaling any one of the three alone runs into
diminishing returns: a deeper network on small images sees too little
detail, and larger images need more layers to cover them with receptive
fields. *Compound scaling* [@tan2019efficientnet] therefore grows all
three together. Each step multiplies the depth by 1.2, the width by 1.1, and
the resolution by 1.15, which about doubles the cost, since
$1.2 \cdot 1.1^2 \cdot 1.15^2 \approx 1.92$. The number of steps is then the
only knob. Three steps, for instance, give a network 1.7 times as deep and
1.3 times as wide, on images 1.5 times as large, at about 7 times the cost.

### Architecture Search

The second question EfficientNet answered with *neural architecture
search*. Its base network was found by an automated search over stacks of
inverted-bottleneck blocks, with an objective that rewards accuracy and
penalizes multiply-adds. Our measurement above shows the weakness of that
objective. EfficientNetV2 [@tan2021efficientnetv2] searched against
measured training speed and parameter count instead, and the search replaced
the depthwise blocks of the early, high-resolution stages by dense
$3 \times 3$ convolutions, which are faster there despite their larger
operation counts. In both networks the block comes from human insight, and
the allocation of width, depth, and resolution to the stages from an
optimization.

### Squeeze-and-Excitation Gates

The inverted bottlenecks that EfficientNet's search stacks carry one more
component after the depthwise convolution, a *squeeze-and-excitation* (SE)
gate [@Hu.Shen.Sun.2018]. A convolution mixes information within a
small window. An SE gate lets the network reweight whole channels on the
evidence of the entire image. It *squeezes* each channel to one number by
global average pooling, passes the resulting vector of $c$ channel summaries
through a two-layer MLP with a hidden width of $c/r$ and a sigmoid output
(the *excitation*), and multiplies each channel of the input by its gate
value in $(0, 1)$. The MLP has about $2c^2/r$ weights for a reduction ratio
$r$ and acts on one vector per image, not on the feature map, so it adds
almost no computation. It is a simple form of attention, computed per
channel rather than per position (that section develops the
general mechanism).

The gate is small and has become common. MobileNetV3 uses it in many of its
blocks, and RegNetY (that section) adds it to every block of a
RegNetX. Our mini-MobileNet does without it, and problem
3 of that section adds it to a
RegNet. The code is a few lines: pool, two dense layers, rescale.

In [ ]:
class SE(tf.keras.Model):
    def __init__(self, num_channels, ratio=4):
        super().__init__()
        self.fc = tf.keras.Sequential([
            tf.keras.layers.Dense(num_channels // ratio, activation='relu',
                                  kernel_initializer=he_init()),
            tf.keras.layers.Dense(num_channels, activation='sigmoid',
                                  kernel_initializer=he_init())])

    def call(self, X):
        s = self.fc(tf.reduce_mean(X, axis=(1, 2)))
        return X * s[:, None, None, :]

SE(32)(tf.random.normal((2, 16, 16, 32))).shape

For $c = 32$ and $r = 4$ the gate has $2 \cdot 32^2 / 4 = 512$ weights plus 40
biases. Its output has the shape of its input, so the gate can be inserted into
any block without changing the rest of the network.

## Structural Re-parameterization

### The RepVGG Block

ResNet (that section) showed that networks with several paths per
layer train better than plain stacks. At inference time, however, every
branch must be computed and its output held in memory until the branches
join, and many small operations leave an accelerator underused. A plain
stack of $3 \times 3$ convolutions maps well to inference hardware: one
large, regular operation per layer and one live activation tensor at a time.
Can we have both? RepVGG [@ding2021repvgg] shows that we can, by
training one network and deploying another. During training, each layer is a
three-branch block: a $3 \times 3$ convolution, a $1 \times 1$ convolution,
and an identity path, each followed by its own batch normalization, summed
before the ReLU (the figure). For deployment, the three
branches are *fused into a single $3 \times 3$ convolution* with the same
output, because at inference time every branch is linear in its input. The
network that is trained has residual paths; the network that is deployed is
a plain stack.

![Structural re-parameterization. At training time each RepVGG layer sums three branches, a $3 \times 3$ convolution, a $1 \times 1$ convolution, and an identity, each with batch normalization. For inference the branches are folded into one $3 \times 3$ convolution with a bias; the two networks compute the same function.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-repvgg-reparam.svg)

In [ ]:
class RepVGGBlock(tf.keras.Model):
    """Train-time block: 3x3, 1x1, and identity branches, each with BN."""
    def __init__(self, num_channels):
        super().__init__()
        self.conv3 = tf.keras.layers.Conv2D(num_channels, 3, padding='same',
                                            use_bias=False)
        self.conv1 = tf.keras.layers.Conv2D(num_channels, 1, use_bias=False)
        self.bn3, self.bn1, self.bn0 = (
            tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5)
            for _ in range(3))

    def call(self, X, training=False):
        return tf.nn.relu(self.bn3(self.conv3(X), training=training)
                          + self.bn1(self.conv1(X), training=training)
                          + self.bn0(X, training=training))

### Fusing the Branches

The fusion takes three steps of weight algebra. First, at inference time
batch normalization is a fixed affine map of each channel: it subtracts the
running mean $\mu_o$, divides by $\sqrt{\sigma_o^2 + \epsilon}$, scales by
$\gamma_o$, and shifts by $\beta_o$. Applied to the output of a convolution
with kernel $\mathbf{W}$ and bias $b_o$, where the index $o$ runs over output
channels, it can be absorbed into a new kernel and bias,

$$
\hat{\mathbf{W}}_o = \frac{\gamma_o}{\sqrt{\sigma_o^2 + \epsilon}}\, \mathbf{W}_o,
\qquad
\hat{b}_o = \beta_o + \frac{\gamma_o(b_o-\mu_o)}{\sqrt{\sigma_o^2 + \epsilon}}.
$$

The kernel of each output channel is rescaled by that channel's
normalization factor, and the shift collects everything that does not depend
on the input. For example, $\gamma_o = 2$, $\sigma_o^2 + \epsilon = 16$,
$\mu_o = 1$, $\beta_o = 1$, and $b_o = 0$ give $\hat{\mathbf{W}}_o =
\mathbf{W}_o / 2$ and $\hat{b}_o = 1 - 0.5 = 0.5$. The convolutions of
`RepVGGBlock` have no bias, so `fuse_bn` below implements the case $b_o = 0$.
Nothing in this step is specific to RepVGG: folding batch normalization into
the preceding convolution is a standard optimization for inference.

Second, the two other branches are convolutions in disguise. A $1 \times 1$
kernel is a $3 \times 3$ kernel that is zero everywhere except at the center,
so padding it with a ring of zeros changes nothing. The identity map is a
$3 \times 3$ convolution whose kernel $\mathbf{W}^{\textrm{id}}_{o,i}$ is 1
at the center position when $i = o$ and 0 otherwise. Third, convolution is
linear in its kernel: three convolutions of the same input, added, equal one
convolution with the sum of the three kernels, and the biases add as well.
Folding each branch by the equation, padding the $1 \times 1$ kernel,
and adding gives the single fused layer (the figure).

![The kernels of one output channel $o$ applied to the input channel $i = o$, after batch normalization has been folded into each branch (illustrative values; after folding, the identity's center entry is $\gamma_o/\sqrt{\sigma_o^2+\epsilon}$ rather than 1). The padded $1 \times 1$ kernel and the identity kernel are nonzero only at the center, so the fused kernel differs from the $3 \times 3$ branch only there. For $i \neq o$ the identity kernel is zero.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-repvgg-kernels.svg)

In [ ]:
def fuse_bn(W, bn):
    """Fold inference-mode batch normalization into a bias-free conv."""
    scale = bn.gamma / tf.sqrt(bn.moving_variance + bn.epsilon)
    return W * scale, bn.beta - bn.moving_mean * scale

def fuse_block(blk):
    c = blk.conv3.kernel.shape[-1]
    pad = [[1, 1], [1, 1], [0, 0], [0, 0]]
    W3, b3 = fuse_bn(blk.conv3.kernel, blk.bn3)
    W1, b1 = fuse_bn(tf.pad(blk.conv1.kernel, pad), blk.bn1)
    I = tf.pad(tf.reshape(tf.eye(c), (1, 1, c, c)), pad)
    W0, b0 = fuse_bn(I, blk.bn0)
    fused = tf.keras.layers.Conv2D(c, 3, padding='same')
    fused.build((None, None, None, c))
    fused.set_weights([W3 + W1 + W0, b3 + b1 + b0])
    return fused

Fusion uses the *running* statistics of batch normalization, the ones
inference uses, so before checking the fused layer we push a few minibatches
with mean 1 and standard deviation 2 through the block in training mode, which
moves those statistics far from their initial values 0 and 1, and we give
the scales $\gamma_o$ and shifts $\beta_o$ random values, so that every term of
the equation matters. The check then feeds fresh data to the block in
inference mode and to the fused convolution and reports the largest
difference.

TensorFlow runs float32 convolutions on recent NVIDIA GPUs in the
reduced-precision TF32 format by default (that section). We turn
TF32 off so that the check measures the fusion, not the rounding of the
convolution.

In [ ]:
tf.config.experimental.enable_tensor_float_32_execution(False)
blk = RepVGGBlock(8)
for _ in range(4):  # Move the BN running statistics
    blk(tf.random.normal((32, 16, 16, 8), 1, 2), training=True)
for bn in (blk.bn3, blk.bn1, blk.bn0):  # Nontrivial scales and shifts
    bn.gamma.assign(tf.random.uniform((8,), 0.5, 1.5))
    bn.beta.assign(tf.random.normal((8,)))
fused = fuse_block(blk)
X = tf.random.normal((32, 16, 16, 8), 1, 2)
float(tf.reduce_max(tf.abs(blk(X, training=False) - tf.nn.relu(fused(X)))))

The largest difference is of order $10^{-6}$, the roundoff of
single-precision arithmetic: the fused convolution computes the same
function. It has $9c^2$ weights against the $10c^2$ of the two branch
convolutions, a saving of a tenth. The larger change is in how the layer
runs: one convolution instead of two convolutions, three normalizations,
and two additions, and one live activation tensor instead of three.

### Quantization after Fusion

Deployment often adds a step that the fusion algebra does not model. To save
memory and energy, inference hardware commonly stores weights and activations
as 8-bit integers, each tensor represented by 256 evenly spaced values. The
fused kernel is a *sum* of branches whose scales, after batch normalization is
folded into them, can differ by orders of magnitude, so the fused weights and
the activations they produce spread over a wide range, and 256 levels resolve
them poorly. The damage can be severe. A fused RepVGG-B0 network quantized
naively to 8 bits after training drops from 75.1% to 40.2% top-1 accuracy on
ImageNet, where a plain ResNet loses a fraction of a point
[@chu2024qarepvgg]. Constraining the branch statistics during training
restores the accuracy. Algebraic equivalence in single-precision floating
point therefore does not imply equivalence after quantization.

## Summary

Efficient convnets treat cost as a design objective alongside accuracy.
Depthwise-separable convolutions cut the arithmetic of a layer to
$1/c_\textrm{o} + 1/k^2$ of a dense convolution, and the inverted bottleneck
arranges them so that the tensors kept between blocks are the narrow ones.
On Fashion-MNIST, a mini-MobileNet comes about two points short of a dense
network with the same number of parameters while performing 7.7 times fewer
multiply-adds, but its forward pass is only 2 to 3.5 times faster, because
depthwise convolutions perform little arithmetic per value they read from
memory. A dense network narrowed to the same arithmetic is more accurate
than MobileNet and faster still: on a task this small, narrow dense layers
are the cheaper choice. Compound scaling and architecture search allocate
depth, width, and resolution once the block is fixed, and searches that
optimize measured latency rather than operation counts avoid depthwise
convolutions where memory traffic dominates. A squeeze-and-excitation gate
lets a block rescale its channels on the evidence of the whole image, for a
negligible share of the arithmetic. Structural re-parameterization trains a multi-branch network and deploys an
equivalent plain stack, with the caveat that equivalence in floating point
need not survive quantization. Each of these choices was made by hand or by
a search for one network at a time; the last section of this chapter asks
how to explore whole families of networks systematically.

## Exercises


1. **Channel permutation between groups.** ● Grouped layers can also be
   mixed without dense weights. Follow a grouped $k \times k$ convolution
   from $c_\textrm{i}$ to $c_\textrm{o}$ channels with $g$ groups by a
   second grouped convolution from $c_\textrm{o}$ to $c_\textrm{o}$
   channels with $g$ groups, and insert a fixed permutation of the
   $c_\textrm{o}$ intermediate channels between the two layers. Find the
   condition on $g$ and $c_\textrm{o}$ under which some permutation makes
   each output of the second layer depend on all $c_\textrm{i}$ input
   channels, and construct such a permutation when the condition holds.

1. [code] **Width multiplier.** Add a width multiplier $m$ to
   `MiniMobileNet` that scales every channel count
   [@howard2017mobilenet]. Predict how the parameter count and the
   number of multiply-adds scale with $m$, and verify your prediction at
   $m = 0.5$ with `num_params` and `mobilenet_macs`.

1. [code] **Folding batch normalization.** The function `fuse_bn` folds
   batch normalization into a preceding convolution without a bias, using
   the equation with $b_o = 0$. Extend `fuse_bn` to a convolution
   with bias $b_o$, and verify numerically that the convolution followed by
   batch normalization in inference mode agrees with the folded convolution.
   Then switch batch normalization to training mode, feed the same example
   inside two different minibatches, and explain why no fixed kernel and
   bias reproduce both outputs.

1. [code] **The inverted bottleneck.** ● MobileNetV2
   [@sandler2018mobilenetv2] replaces the block of the figure
   by an inverted bottleneck: a $1 \times 1$ expansion by a factor $t$, a
   depthwise $3 \times 3$ convolution, a linear $1 \times 1$ projection, and
   a residual connection whenever input and output shapes match.
    1. Implement the block with $t = 6$ and replace the blocks of
       `MiniMobileNet` by it, choosing the widths so that the parameter count
       stays near 542,474. Train it with the setup of
       the table, and compare its validation accuracy and its
       forward time with those of `MiniMobileNet` on the same device.
    1. Add a ReLU after the projection. Train the linear and the ReLU
       variants three times each, and compare the gap in validation accuracy
       with the spread across runs. Does the paper's ablation, which favors
       the linear projection, reproduce at this scale?
    1. For a $96 \times 96$ input, tabulate for each block of both networks
       the number of elements in its input, its output, and its largest
       internal tensor. Which network needs more activation memory when
       every tensor is held in full? Show that an inference implementation
       of the inverted bottleneck never needs to hold a whole expanded
       tensor, and determine which tensors then set its peak activation
       memory.

1. [code] **Width versus resolution.** ● Lowering the input resolution is a
   second way to cut multiply-adds. Using the width multiplier of
   problem 2, train
   `MiniMobileNet` at $m = 1$ on $48 \times 48$ inputs and at
   $m = 0.5$ on $96 \times 96$ inputs, twice each and otherwise with the
   setup of the table. Compare the two networks in
   multiply-adds, parameter count, and validation accuracy. Fashion-MNIST
   images have $28 \times 28$ pixels. How does this limit what the
   comparison says
   about scaling resolution in EfficientNet [@tan2019efficientnet]?


1. [code] **Arithmetic intensity.** ● `MiniMobileNet` performs 7.7 times
   fewer multiply-adds than `VGGSmall`, yet its forward pass in
   the table is only 2 to 3.5 times faster.
    1. The arithmetic intensity of a layer is its number of multiply-adds
       divided by the number of values it reads and writes: weights, input
       activations, and output activations. For every block of
       `MiniMobileNet`, compute it for the depthwise convolution, for the
       pointwise convolution, and for a dense $3 \times 3$ convolution with
       the block's input width, output width, and stride. In which stages
       is the ratio between the dense and the depthwise intensity largest?
       Compare them with the stages in which EfficientNetV2
       [@tan2021efficientnetv2] replaces depthwise blocks by dense
       ones.
    1. Measure `forward_ms` for both networks at batch sizes 1, 16, 128,
       and 512, and plot the time per image against the batch size. How
       does the ratio of the two networks' times change with the batch
       size, and why?

1. [code] **Normalization before the convolution.** ● Let batch
   normalization come before a $3 \times 3$ convolution with zero padding of
   1, with no nonlinearity in between. Fold it into the kernel and bias of
   the convolution and compare the folded layer with the original
   numerically. At which output positions do they disagree, and with what
   value would the input have to be padded for the fold to be exact?

1. [code] **Training with RepVGG blocks.** ● If the fused RepVGG layer
   computes the same function, why not train the plain $3 \times 3$ stack
   directly?
    1. Give `RepVGGBlock` a stride and an output width. A downsampling
       block has stride 2 and a new width, so an identity branch cannot
       match its shape. Such a block keeps only its $3 \times 3$ and
       $1 \times 1$ branches [@ding2021repvgg]. Build a small network
       from these blocks and a plain network that replaces each block by one
       $3 \times 3$ convolution of the same width and stride, followed by
       batch normalization and a ReLU. Train both on Fashion-MNIST and
       compare the training curves.
    1. Extend `fuse_block` to the downsampling blocks, fuse the trained
       RepVGG network, and confirm that its validation accuracy is unchanged.
    1. Fold the batch normalization of the plain network into its
       convolutions with `fuse_bn`. Then quantize the weights of both
       deployed networks to eight bits with a symmetric per-tensor scale,
       $\hat{w} = s \cdot \mathrm{round}(w / s)$ with $s = \max |w| / 127$,
       and compare the accuracy drops. Relate the outcome to the
       distribution of the fused weights, as @chu2024qarepvgg
       analyze.

[Discussions](https://d2l.discourse.group/t/36760)